In [2]:
# =========================================================
# CELL 1 — Imports + Reproducibility
# =========================================================

import os
import re
import math
import ast
import json
import pickle
import random
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# HuggingFace (decoder for generation head)
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Prevent NameError later (some cells reference df_all as a bare expression)
df_all = None


In [3]:
# =========================================================
# CELL 1.5 — Device
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [4]:
# =========================================================
# CELL 2 — Load CSVs + Embeddings
# =========================================================

MAIN_PATH   = r"/teamspace/studios/this_studio/datasets/main_mixed_df.csv"
DIALOG_PATH = r"/teamspace/studios/this_studio/datasets/dialogue.csv"
DOCS_PATH   = r"/teamspace/studios/this_studio/datasets/documents.csv"
RESP_PATH   = r"/teamspace/studios/this_studio/datasets/response.csv"
SUMM_PATH   = r"/teamspace/studios/this_studio/datasets/summaries.csv"

EMB_DIALOG_PATH = r"/teamspace/studios/this_studio/embeddings/dialog_embeddings.pkl"
EMB_DOCS_PATH   = r"/teamspace/studios/this_studio/embeddings/doc_embeddings.pkl"
EMB_RESP_PATH   = r"/teamspace/studios/this_studio/embeddings/response_embeddings.pkl"
EMB_SUMM_PATH   = r"/teamspace/studios/this_studio/embeddings/summ_embeddings.pkl"

main_df   = pd.read_csv(MAIN_PATH)
dialog_df = pd.read_csv(DIALOG_PATH)
docs_df   = pd.read_csv(DOCS_PATH)
resp_df   = pd.read_csv(RESP_PATH)
summ_df   = pd.read_csv(SUMM_PATH)

def load_embeddings(file_path):
    with open(file_path, "rb") as f:
        return pickle.load(f)

emb_dialog = load_embeddings(EMB_DIALOG_PATH)
emb_docs   = load_embeddings(EMB_DOCS_PATH)
emb_resp   = load_embeddings(EMB_RESP_PATH)
emb_summ   = load_embeddings(EMB_SUMM_PATH)

print("Loaded main_df columns:", list(main_df.columns))
print("Rows:", len(main_df))
print("Sample row:\n", main_df.iloc[0])


Loaded main_df columns: ['Unnamed: 0', 'userid', 'target_user_id', 'task', 'text_nodes', 'interactions', 'query_text_id', 'target_text_id', 'split']
Rows: 113274
Sample row:
 Unnamed: 0                                                        0
userid                                                      U203312
target_user_id                                              U203312
task                                           next_item_prediction
text_nodes        ['d-100554', 'd-114190', 'd-58081', 'd-40294',...
interactions      ['pos', 'pos', 'pos', 'pos', 'neg', 'pos', 'po...
query_text_id                                               d-57729
target_text_id                                              d-57729
split                                                         train
Name: 0, dtype: object


In [5]:
# =========================================================
# CELL 3 — Core Helpers (sequence parse, canonical ids, null checks)
# =========================================================

CANON_RE = re.compile(r"^(dl|d|r|s)-\d+$", re.IGNORECASE)

def is_nullish(x):
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip().lower()
    return s in {"", "null", "none", "nan"}

def normalize_id(x):
    return str(x).strip()

def parse_serialized_sequence(x):
    """
    Accepts:
      - Python list
      - '["a","b"]'
      - '[a,b]'
    Returns: list[str]
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip() != ""]
    s = str(x).strip()
    if not (s.startswith("[") and s.endswith("]")):
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [str(t).strip() for t in v if str(t).strip() != ""]
    except Exception:
        pass
    items = [t.strip().strip('"').strip("'") for t in s[1:-1].split(",")]
    return [t for t in items if t]

def fix_and_align_sequences(nodes, actions, max_len=100):
    nodes = list(nodes)
    actions = list(actions)
    L = min(len(nodes), len(actions), max_len)
    return nodes[:L], actions[:L]


In [6]:
# =========================================================
# CELL 4 — Build id2text + id2phrases (canonical IDs, NO double-prefix)
#   - If source id already looks like -, keep it.
#   - Else add the required prefix.
#   - id2phrases == id2text (string), as requested.
# =========================================================

from collections import defaultdict
import re

CANON_RE = re.compile(r"^(dl|d|r|s)-\d+$", re.IGNORECASE)

def _norm_id(x):
    return "" if x is None else str(x).strip()

def _canonicalize(prefix: str, raw_id: str):
    rid = _norm_id(raw_id)
    if rid == "":
        return ""
    # If already canonical like dl-123 / d-456 / r-7 / s-9, keep as-is.
    if CANON_RE.match(rid):
        return rid.lower()
    # If it starts with prefix + "-", also keep (covers non-numeric ids too)
    if rid.lower().startswith(prefix.lower() + "-"):
        return rid.lower()
    return f"{prefix.lower()}-{rid}"

def _join_head_body(head, body):
    h = "" if head is None else str(head).strip()
    b = "" if body is None else str(body).strip()
    if h == "" and b == "":
        return ""
    if h != "" and b != "":
        return h + "\n" + b
    return h if h != "" else b

def _add_map(id2text, prefix, ids, texts):
    for rid, txt in zip(ids, texts):
        key = _canonicalize(prefix, rid)
        if key == "":
            continue
        if key not in id2text:
            id2text[key] = "" if txt is None else str(txt)

# --------- Hard checks for required columns ----------
required = {
    "dialog_df": (dialog_df, ["dialogue_id", "dialogue_text"]),
    "resp_df":   (resp_df,   ["response_id", "response_text"]),
    "summ_df":   (summ_df,   ["SummID", "Summary"]),
    "docs_df":   (docs_df,   ["NewsID", "Headline", "NewsBody"]),
}
for name, (df, cols) in required.items():
    missing = [c for c in cols if c not in df.columns]
    if len(missing) > 0:
        raise KeyError(f"{name} missing columns: {missing}. Present columns: {list(df.columns)}")

# --------- Build id2text ----------
id2text = defaultdict(str)

# dialog: dl- (but don't double-prefix if already dl-*)
_add_map(
    id2text,
    prefix="dl",
    ids=dialog_df["dialogue_id"].tolist(),
    texts=dialog_df["dialogue_text"].tolist(),
)

# response: r-
_add_map(
    id2text,
    prefix="r",
    ids=resp_df["response_id"].tolist(),
    texts=resp_df["response_text"].tolist(),
)

# summary: s-
_add_map(
    id2text,
    prefix="s",
    ids=summ_df["SummID"].tolist(),
    texts=summ_df["Summary"].tolist(),
)

# docs: d- where text = Headline + "\n" + NewsBody
docs_text = [
    _join_head_body(h, b)
    for h, b in zip(docs_df["Headline"].tolist(), docs_df["NewsBody"].tolist())
]
_add_map(
    id2text,
    prefix="d",
    ids=docs_df["NewsID"].tolist(),
    texts=docs_text,
)

print("id2text size:", len(id2text))

# --------- id2phrases is same as id2text (string) ----------
id2phrases = defaultdict(str)
for k, v in id2text.items():
    id2phrases[k] = v

print("id2phrases size:", len(id2phrases))

# --------- sanity samples ----------
for k in list(id2text.keys())[:5]:
    print("\nSAMPLE KEY:", k)
    print(id2text[k][:200])


id2text size: 177454
id2phrases size: 177454

SAMPLE KEY: dl-1
super cute pup! lol where do our pups get their giant ears from? :d

SAMPLE KEY: dl-2
unfortunately no, she was a stray. the shelter said she's most likely lab/shepherd. our vet says she's got some pit in her too. but in the end who knows.

SAMPLE KEY: dl-3
i love those. but my knitting will be slowed down by 100% because i would be too busy admiring the bears and forgetting what i was doing in the first place

SAMPLE KEY: dl-4
i want to wear those as earrings! adorable

SAMPLE KEY: dl-5
omg i just bought these and the earrings because i couldn’t get over how cute they are so reasonably priced


In [7]:
# =========================================================
# CELL 5 — Embedding Resolver for canonical ids
# =========================================================

class EmbLookup:
    @staticmethod
    def lookup(emb_dict, key):
        if key in emb_dict:
            return emb_dict[key]
        s = str(key)
        if s in emb_dict:
            return emb_dict[s]
        nums = re.findall(r"\d+", s)
        if nums:
            n = nums[-1]
            if n in emb_dict:
                return emb_dict[n]
            try:
                ni = int(n)
                if ni in emb_dict:
                    return emb_dict[ni]
            except Exception:
                pass
        return None

class CanonicalEmbeddingResolver:
    def __init__(self, emb_dialog, emb_docs, emb_resp, emb_summ, d=768):
        self.emb_dialog = emb_dialog
        self.emb_docs = emb_docs
        self.emb_resp = emb_resp
        self.emb_summ = emb_summ
        self.d = d

    def get(self, cid: str) -> np.ndarray:
        cid = normalize_id(cid)
        prefix = cid.split("-")[0].lower()

        if prefix == "dl":
            v = EmbLookup.lookup(self.emb_dialog, cid)
        elif prefix == "d":
            v = EmbLookup.lookup(self.emb_docs, cid)
        elif prefix == "r":
            v = EmbLookup.lookup(self.emb_resp, cid)
        elif prefix == "s":
            v = EmbLookup.lookup(self.emb_summ, cid)
        else:
            v = None

        if v is None:
            return np.zeros((self.d,), dtype=np.float32)

        v = np.asarray(v, dtype=np.float32)
        if v.shape[-1] != self.d:
            raise ValueError(f"Embedding dim mismatch for {cid}: got {v.shape}, expected ({self.d},)")
        return v

D_MODEL = 768
emb_resolver = CanonicalEmbeddingResolver(emb_dialog, emb_docs, emb_resp, emb_summ, d=D_MODEL)
print("Embedding resolver ready.")


Embedding resolver ready.


In [8]:
# =========================================================
# CELL 6 — Task Names + Task IDs
# =========================================================

assert "task" in main_df.columns, "main_df must contain 'task'"

task_names = sorted(main_df["task"].dropna().astype(str).unique())
task2idx = {t: i for i, t in enumerate(task_names)}
idx2task = {i: t for t, i in task2idx.items()}
main_df["task_id"] = main_df["task"].astype(str).map(task2idx)

assert main_df["task_id"].isna().sum() == 0, "Unmapped task found"

n_tasks = len(task2idx)
print("Tasks:", task2idx)


Tasks: {'dialogue_generation': 0, 'next_item_prediction': 1, 'next_user_prediction': 2, 'personalized_summarization': 3}


In [9]:
# =========================================================
# CELL 7 — ACTION vocab (all rows) and NODE vocab (all rows)
# =========================================================

all_actions = set()
for seq in main_df["interactions"].dropna():
    all_actions.update(parse_serialized_sequence(seq))

actions2idx = {a: i for i, a in enumerate(sorted(all_actions))}
idx2action  = {i: a for a, i in actions2idx.items()}

node_ids = set()
for seq in main_df["text_nodes"].dropna():
    for nid in parse_serialized_sequence(seq):
        if not is_nullish(nid):
            node_ids.add(normalize_id(nid))

for col in ["query_text_id", "target_text_id"]:
    if col in main_df.columns:
        for x in main_df[col].dropna():
            if not is_nullish(x):
                node_ids.add(normalize_id(x))

node2idx = {nid: i for i, nid in enumerate(sorted(node_ids))}
idx2node = {i: nid for nid, i in node2idx.items()}

print("VOCAB SIZES:")
print("  actions:", len(actions2idx))
print("  nodes  :", len(node2idx))


VOCAB SIZES:
  actions: 4
  nodes  : 90496


In [10]:
# =========================================================
# CELL 8 — USER vocab ONLY from next_user_prediction rows
# =========================================================

USER_TASK = "next_user_prediction"
ITEM_TASK = "next_item_prediction"

user_rows = main_df[main_df["task"].astype(str).str.strip() == USER_TASK]
unique_users_small = sorted(user_rows["userid"].dropna().astype(str).unique())
unique_users_small = [""] + [u for u in unique_users_small if u != ""]

user2idx_small = {u: i for i, u in enumerate(unique_users_small)}
idx2user_small = {i: u for u, i in user2idx_small.items()}

print("User vocab size (next_user_prediction only):", len(user2idx_small))


User vocab size (next_user_prediction only): 4527


In [11]:
# =========================================================
# CELL 9 — Train/Test Split
# =========================================================

assert "split" in main_df.columns, "main_df must contain 'split' with train/test."
main_df["split"] = main_df["split"].astype(str).str.strip().str.lower()

train_df = main_df[main_df["split"] == "train"].reset_index(drop=True)
test_df  = main_df[main_df["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df), "| Test size:", len(test_df))
print("Task counts (train):\n", train_df["task"].value_counts())
print("Task counts (test):\n", test_df["task"].value_counts())


Train size: 78274 | Test size: 35000
Task counts (train):
 task
next_item_prediction          34630
personalized_summarization    34630
next_user_prediction           4507
dialogue_generation            4507
Name: count, dtype: int64
Task counts (test):
 task
next_item_prediction          15000
personalized_summarization    15000
next_user_prediction           2500
dialogue_generation            2500
Name: count, dtype: int64


In [12]:
# =========================================================
# CELL 10 — Tokenizer + Decoder (Distil GPT / MiniGPT via HF)
#   - If weights are not cached locally, HF will attempt to download.
#   - If download is not possible in your environment, this will error.
# =========================================================

DECODER_NAME = "distilgpt2"   # change to a locally-cached minigpt-like causal LM if you have one

try:
    tok = AutoTokenizer.from_pretrained(DECODER_NAME, use_fast=True)
    dec = AutoModelForCausalLM.from_pretrained(DECODER_NAME)
except Exception as e:
    raise RuntimeError(
        f"Could not load decoder '{DECODER_NAME}'. "
        f"Ensure model weights are available (cached or downloadable). "
        f"Original error: {repr(e)}"
    )

# GPT2 family typically has no pad token -> set pad to eos for batching
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

dec = dec.to(device)
dec.eval()

DEC_H = dec.config.n_embd
print("Decoder hidden size:", DEC_H)


Decoder hidden size: 768


In [13]:
# =========================================================
# CELL 11 — Batch Builder (row -> tensors) with TASK masks
#   - User head supervised ONLY for USER_TASK
#   - Item head supervised ONLY for ITEM_TASK using target_text_id (tid) mapped into node2idx
#   - Generation head supervised ONLY for generation tasks using id2phrases[tid]
#   - Encoder (SSM) losses always computed, but can be weighted if desired
# =========================================================

MAX_T = 60
IGNORE_INDEX = -100

GEN_TASKS = set([t for t in task_names if ("summar" in t.lower()) or ("dialogue_generation" in t.lower())])

# If generation tasks exist in the data, require id2phrases coverage to be non-empty
if any((t in GEN_TASKS) for t in task_names):
    if len(id2phrases) == 0:
        raise ValueError(
            "Generation tasks exist, but id2phrases is empty. "
            "Add a phrases/keyphrases/keywords column to one of the source CSVs "
            "so infer_id_phrases_map can build id2phrases."
        )

def safe_user_label(userid: str):
    userid = str(userid).strip()
    if userid in user2idx_small:
        return user2idx_small[userid]
    return 0

def safe_node_label(text_id: str):
    text_id = normalize_id(text_id)
    if text_id in node2idx:
        return node2idx[text_id]
    return 0

def phrases_to_text(x):
    # You are storing FULL TEXT in id2phrases (string), so return as-is.
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()

    # Backward-compatible: if ever you store a list of phrases
    if isinstance(x, (list, tuple)):
        xs = [str(p).strip() for p in x if str(p).strip()]
        return "; ".join(xs).strip()

    return str(x).strip()

def build_gen_tensors(tid: str, device, max_len=128):
    x = id2phrases.get(tid, "")
    gen_text = phrases_to_text(x)

    if gen_text == "":
        return None, None, None, "empty_gen_target"

    enc = tok(
        gen_text,
        truncation=True,
        max_length=max_len,
        padding=False,
        return_tensors="pt"
    )
    input_ids = enc["input_ids"]
    attn = enc["attention_mask"]

    if input_ids.size(1) < 2:
        return None, None, None, "too_short_gen_target"

    return input_ids.to(device), attn.to(device), gen_text, None

def build_row_tensors(row, device):
    task = str(row["task"]).strip()
    task_id = int(row["task_id"])

    actions = parse_serialized_sequence(row["interactions"])
    nodes   = parse_serialized_sequence(row["text_nodes"])

    if len(actions) == 0 or len(nodes) == 0:
        return {"ok": False, "reason": "empty_sequence", "task": task}

    nodes, actions = fix_and_align_sequences(nodes, actions, max_len=MAX_T)
    if len(nodes) < 3:
        return {"ok": False, "reason": "too_short_sequence", "task": task}

    T = len(nodes)

    try:
        action_ids = [actions2idx[a] for a in actions]
    except KeyError:
        return {"ok": False, "reason": "unknown_action_token", "task": task}

    action_labels_full = torch.tensor(action_ids, device=device).unsqueeze(0)  # (1,T)
    action_onehot = F.one_hot(action_labels_full, num_classes=len(actions2idx)).float()  # (1,T,A)

    nodes_norm = [normalize_id(n) for n in nodes]
    try:
        node_ids = [node2idx[n] for n in nodes_norm]
    except KeyError:
        return {"ok": False, "reason": "unknown_node_token", "task": task}

    node_labels_full = torch.tensor(node_ids, device=device).unsqueeze(0)  # (1,T)

    e_seq = torch.tensor(
        np.stack([emb_resolver.get(n) for n in nodes_norm], axis=0),
        device=device
    ).unsqueeze(0)  # (1,T,768)

    qid = normalize_id(row["query_text_id"]) if "query_text_id" in row else nodes_norm[-2]
    tid = normalize_id(row["target_text_id"]) if "target_text_id" in row else nodes_norm[-1]

    q_text_emb = torch.tensor(emb_resolver.get(qid), device=device).unsqueeze(0)  # (1,768)

    task_labels = torch.tensor([task_id], device=device)

    # ===== Encoder supervision masking logic (NO item bug):
    # - Encoder processes only 0..T-3 internally (T-2 steps).
    # - We typically want to ignore index T-2 (second-last) for both action/node (query step),
    #   but keep index T-1 as the last-step supervised target when available.
    action_labels = action_labels_full.clone()
    node_labels   = node_labels_full.clone()

    if T >= 2:
        # ignore the second-last position (T-2 index), keep last (T-1 index)
        action_labels[0, T-2] = IGNORE_INDEX
        node_labels[0, T-2]   = IGNORE_INDEX

    # ===== Task masks + labels for heads
    user_mask = torch.tensor([1.0 if task == USER_TASK else 0.0], device=device)
    user_labels = torch.tensor([safe_user_label(row["userid"])], device=device)

    item_mask = torch.tensor([1.0 if task == ITEM_TASK else 0.0], device=device)
    item_labels = torch.tensor([safe_node_label(tid)], device=device)

    gen_mask = torch.tensor([1.0 if task in GEN_TASKS else 0.0], device=device)
    gen_input_ids, gen_attn, gen_text, gen_err = None, None, "", None
    if task in GEN_TASKS:
        gen_input_ids, gen_attn, gen_text, gen_err = build_gen_tensors(tid, device)
        if gen_err is not None:
            return {"ok": False, "reason": f"gen_{gen_err}", "task": task}

    return {
        "ok": True,
        "task": task,
        "T": T,

        "e_seq": e_seq,
        "action_onehot": action_onehot,
        "action_labels": action_labels,
        "node_labels": node_labels,

        "task_labels": task_labels,
        "q_text_emb": q_text_emb,

        "user_labels": user_labels,
        "user_mask": user_mask,

        "item_labels": item_labels,
        "item_mask": item_mask,

        "gen_mask": gen_mask,
        "gen_input_ids": gen_input_ids,   # (1,L) or None
        "gen_attn": gen_attn,             # (1,L) or None
        "gen_text": gen_text,

        "qid": qid,
        "tid": tid,
        "userid_raw": str(row["userid"]).strip(),
    }


In [14]:
# =========================================================
# CELL 12 — FocalCELoss (encoder uses it for task)
# =========================================================

class FocalCELoss(nn.Module):
    def __init__(self, gamma=2.0, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


In [15]:
# =========================================================
# CELL 13 — UniHis SSM Encoder (MODIFIED: Mamba-style diagonalizable stable A)
#   Change (2): A making now uses a Mamba-style stable eigenvalue parameterization:
#     - Q is treated as an eigenbasis and made orthonormal via QR (ensures invertibility).
#     - Continuous-time diagonal eigenvalues are negative: A_ct = -exp(Lambda) (strictly < 0).
#     - Discrete-time stable eigenvalues are V = exp(dt * A_ct) in (0, 1), guaranteeing |eig| < 1.
#     - Transition is A = Q diag(V) Q^{-1} with Q^{-1}=Q^T (orthonormal basis).
#   All other behavior is preserved.
# =========================================================

class UniHisSSMEncoder(nn.Module):
    def __init__(
        self,
        vocab_sizes,
        d_model=768,
        d_state=64,
        n_tasks=8,
        d_qtext=768,
        known_task=False
    ):
        super().__init__()
        self.d_state = d_state
        self.known_task = known_task

        self.criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
        self.task_criterion = FocalCELoss(gamma=2.0, reduction="mean")

        def init_eig():
            # Q is an unconstrained basis; we will orthonormalize it via QR at runtime.
            Q = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
            # Lambda stores log-rates (unconstrained); used as A_ct_diag = -exp(Lambda) < 0.
            Lambda = nn.Parameter(torch.empty(d_state).uniform_(-1.0, -0.1))
            return Q, Lambda

        self.Q_L, self.Lambda_L = init_eig()
        self.Q_S, self.Lambda_S = init_eig()
        self.Q_E, self.Lambda_E = init_eig()

        self.B = nn.Parameter(torch.randn(d_state, d_model))

        self.tau_S = nn.Parameter(torch.tensor(1.0))
        self.tau_E = nn.Parameter(torch.tensor(1.0))

        self.action_proj = nn.Linear(vocab_sizes["actions"], d_model)
        self.MLP_mood = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )
        self.W_a = nn.Linear(d_model, d_state)

        self.Mixer_L = nn.Linear(d_state, d_state)
        self.Mixer_S = nn.Linear(d_state, d_state)
        self.Mixer_E = nn.Linear(d_state, d_state)

        self.W_fuse = nn.Linear(3 * d_state, 3)
        self.Mixer_Fuse = nn.Linear(d_state, d_state)

        self.action_head = nn.Linear(3 * d_state, vocab_sizes["actions"])
        self.node_head   = nn.Linear(3 * d_state, vocab_sizes["nodes"])

        self.Embed_Q = nn.Linear(d_qtext, d_state)
        self.W_task  = nn.Linear(2 * d_state, n_tasks)

        self.W_hyp = nn.Linear(2 * d_state, d_state)
        self.W_theta = nn.ModuleList([
            nn.Linear(d_state, 2 * d_state) for _ in range(n_tasks)
        ])

        self.W_step = nn.Linear(d_state, d_state)
        self.MLP_step = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.ReLU(),
            nn.Linear(d_state, d_state)
        )

        self.action_last_head = nn.Linear(d_state, vocab_sizes["actions"])
        self.node_last_head   = nn.Linear(d_state, vocab_sizes["nodes"])

    def make_A(self, Q_raw, Lambda_log_rate, dt=1.0, eps=1e-4):
        """
        Mamba-style diagonalizable stable discrete transition:
          - Orthonormal eigenbasis Q via QR => Q^{-1} = Q^T
          - Continuous-time diag: A_ct = -exp(Lambda_log_rate) < 0 (strict)
          - Discrete-time stable diag: V = exp(dt * A_ct) in (0,1]
          - Enforce strict < 1 by scaling with (1-eps)
          - A = Q diag(V) Q^T
        """
        # Orthonormalize basis to guarantee invertibility and numerical stability
        Q, _ = torch.linalg.qr(Q_raw)

        # Ensure dt is scalar tensor on the correct device/dtype
        dt_t = torch.as_tensor(dt, device=Q.device, dtype=Q.dtype)
        if dt_t.ndim > 0:
            dt_t = dt_t.reshape(())

        # Continuous-time negative rates
        rates = torch.exp(Lambda_log_rate.to(dtype=Q.dtype))
        A_ct_diag = -rates  # strictly negative

        # Discretization: stable eigenvalues in (0, 1]
        V = torch.exp(dt_t * A_ct_diag)

        # Strictness: keep strictly inside unit circle
        V = V * (1.0 - float(eps))

        # Construct A
        A = Q @ torch.diag(V) @ Q.T
        return A

    def forward(
        self,
        e_seq,             # (B, T, d_model)
        action_onehot,     # (B, T, |A|)
        action_labels,     # (B, T) (may contain IGNORE_INDEX)
        node_labels,       # (B, T) (may contain IGNORE_INDEX)
        task_labels,       # (B,)
        q_text_emb         # (B, d_qtext)
    ):
        B, T, _ = e_seq.shape
        device = e_seq.device

        # processed length = T-2 => indices 0..T-3
        T_proc = max(0, T - 2)

        xL = torch.zeros(B, self.d_state, device=device)
        xS = torch.zeros(B, self.d_state, device=device)
        xE = torch.zeros(B, self.d_state, device=device)

        act_logits_proc = []
        node_logits_proc = []

        # -----------------------------
        # Mamba-style stable transitions
        # -----------------------------
        # Long: fixed dt=1
        A_L = self.make_A(self.Q_L, self.Lambda_L, dt=1.0)

        # Short: derive a positive dt from tau_S; keep prior computations and use them to set dt
        eps_S = torch.sigmoid(self.tau_S)
        # Keep k computation (previously used) but repurpose into a stable, positive dt scalar
        # Add numerical safeguards for division
        denom_S = torch.mean(torch.abs(self.Lambda_S)) + 1e-6
        k = torch.log((1 - eps_S) / (eps_S + 1e-12)) / denom_S
        dt_S = torch.clamp(torch.abs(k), min=1e-4, max=10.0)
        A_S = self.make_A(self.Q_S, self.Lambda_S, dt=dt_S)

        # Event: keep eta computation (previously used) but repurpose into a stable, positive dt scalar
        tauE_pos = F.softplus(self.tau_E) + 1e-6
        eta = (torch.exp(-1.0 / tauE_pos) - 1.0) / (self.Lambda_E + 1e-6)
        dt_E = torch.clamp(torch.mean(torch.abs(eta)), min=1e-4, max=10.0)
        A_E = self.make_A(self.Q_E, self.Lambda_E, dt=dt_E)

        for t in range(T_proc):
            e_t = e_seq[:, t]
            a_t = action_onehot[:, t, :]

            a_emb = self.action_proj(a_t)
            mood_vec = self.MLP_mood(a_emb)
            mood = torch.sigmoid(self.W_a(mood_vec))

            u_t = torch.bmm(
                (self.B.unsqueeze(0) * mood.unsqueeze(2)),
                e_t.unsqueeze(2)
            ).squeeze(2)

            xL = torch.matmul(xL, A_L.T) + u_t
            xS = torch.matmul(xS, A_S.T) + u_t
            xE = torch.matmul(xE, A_E.T) + u_t

            x_cat = torch.cat([xL, xS, xE], dim=-1)
            act_logits_proc.append(self.action_head(x_cat))
            node_logits_proc.append(self.node_head(x_cat))

        act_logits_proc  = torch.stack(act_logits_proc, dim=1)     # (B, T-2, |A|)
        node_logits_proc = torch.stack(node_logits_proc, dim=1)    # (B, T-2, |V|)

        yL = self.Mixer_L(xL)
        yS = self.Mixer_S(xS)
        yE = self.Mixer_E(xE)

        w = F.softmax(self.W_fuse(torch.cat([xL, xS, xE], dim=-1)), dim=-1)
        h_hist = self.Mixer_Fuse(
            w[:, 0:1] * yL +
            w[:, 1:2] * yS +
            w[:, 2:3] * yE
        )

        q_emb = self.Embed_Q(q_text_emb)
        z_switch = torch.cat([h_hist, q_emb], dim=-1)

        task_logits = self.W_task(z_switch)
        P_task = F.softmax(task_logits, dim=-1)
        loss_task = self.task_criterion(task_logits, task_labels)

        h_hyp = F.relu(self.W_hyp(z_switch))

        if self.known_task:
            theta = self.W_theta[task_labels[0]](h_hyp)
            g, b = theta.chunk(2, dim=-1)
            h_final_minus2 = torch.sigmoid(g) * h_hist + b
        else:
            h_final_minus2 = 0.0
            for i in range(len(self.W_theta)):
                theta_i = self.W_theta[i](h_hyp)
                g_i, b_i = theta_i.chunk(2, dim=-1)
                h_final_minus2 = h_final_minus2 + P_task[:, i:i+1] * (
                    torch.sigmoid(g_i) * h_hist + b_i
                )

        h_final_minus1 = self.W_step(h_final_minus2)
        h_final = self.MLP_step(h_final_minus1)

        action_labels_proc = action_labels[:, :T_proc]
        node_labels_proc   = node_labels[:, :T_proc]

        loss_action_proc = self.criterion(
            act_logits_proc.reshape(-1, act_logits_proc.size(-1)),
            action_labels_proc.reshape(-1)
        )
        loss_node_proc = self.criterion(
            node_logits_proc.reshape(-1, node_logits_proc.size(-1)),
            node_labels_proc.reshape(-1)
        )

        last_action_label = action_labels[:, -1]
        last_node_label   = node_labels[:, -1]

        act_last_logits = self.action_last_head(h_final)
        node_last_logits= self.node_last_head(h_final)

        loss_action_last = torch.tensor(0.0, device=device)
        loss_node_last   = torch.tensor(0.0, device=device)

        valid_act = (last_action_label != IGNORE_INDEX)
        valid_node = (last_node_label != IGNORE_INDEX)

        if valid_act.any():
            loss_action_last = F.cross_entropy(act_last_logits[valid_act], last_action_label[valid_act])
        if valid_node.any():
            loss_node_last = F.cross_entropy(node_last_logits[valid_node], last_node_label[valid_node])

        loss_action = loss_action_proc + loss_action_last
        loss_node   = loss_node_proc + loss_node_last

        return {
            "loss_action": loss_action,
            "loss_node": loss_node,
            "loss_task": loss_task,

            "task_logits": task_logits,
            "P_task": P_task,

            "h_final_minus2": h_final_minus2,
            "h_final_minus1": h_final_minus1,
            "h_final": h_final
        }


In [16]:
# =========================================================
# CELL 14 — Heads: User (as-is), Item MLP, Generator Bridge
#   Change (1): Bridge now acts as a FiLM-style gate:
#     - query text embedding (q_text_emb) produces (gamma, beta)
#     - gated representation: h_g = (sigmoid(gamma)*2)*h + beta, then LayerNorm
#     - prefix adapter uses gated h_g
#   All other functionality preserved.
# =========================================================

class UserBilinearRanker(nn.Module):
    def __init__(self, d_state, n_users):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
        self.U = nn.Embedding(n_users, d_state)
        nn.init.normal_(self.U.weight, std=0.02)

    def scores_all(self, h):
        return (h @ self.W) @ self.U.weight.T

    def scores_subset(self, h, user_ids):
        hw = h @ self.W
        Usub = self.U(user_ids)
        return torch.einsum("bd,bkd->bk", hw, Usub)

def sampled_user_loss(ranker, h, gold_users, n_neg=64):
    B = h.size(0)
    device = h.device
    neg = torch.randint(0, ranker.U.num_embeddings, (B, n_neg), device=device)
    gold = gold_users.unsqueeze(1)
    neg = torch.where(neg == gold, (neg + 1) % ranker.U.num_embeddings, neg)
    cand = torch.cat([gold, neg], dim=1)
    logits = ranker.scores_subset(h, cand)
    y = torch.zeros(B, dtype=torch.long, device=device)
    return F.cross_entropy(logits, y)

class ItemMLPHead(nn.Module):
    def __init__(self, d_state, n_nodes, hidden=256, dropout=0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_state, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_nodes)
        )

    def forward(self, h):
        return self.mlp(h)  # (B, n_nodes)

class BridgeAdapter(nn.Module):
    """
    FiLM-gated + prefix-style adapter:

    1) FiLM gate:
       q_text_emb -> (gamma, beta) in R^{d_state}
       h_g = LN( (sigmoid(gamma)*2) * h + beta )

    2) Prefix adapter for decoder:
       h_g -> (B, n_prefix, dec_hidden), concatenated with token embeddings for generation
    """
    def __init__(self, d_state, dec_hidden, d_qtext=768, n_prefix=8):
        super().__init__()
        self.n_prefix = n_prefix
        self.d_state = d_state
        self.d_qtext = d_qtext

        # FiLM parameters from query embedding
        self.film = nn.Linear(d_qtext, 2 * d_state)
        self.h_norm = nn.LayerNorm(d_state)

        # Prefix projection from gated h
        self.proj = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.ReLU(),
            nn.Linear(d_state, n_prefix * dec_hidden)
        )

    def gate(self, h, q_text_emb):
        """
        h: (B, d_state)
        q_text_emb: (B, d_qtext)
        returns: h_gated: (B, d_state)
        """
        gb = self.film(q_text_emb)
        g, b = gb.chunk(2, dim=-1)

        # scale in (0, 2) for stable FiLM gating
        scale = torch.sigmoid(g) * 2.0

        h_gated = scale * h + b
        h_gated = self.h_norm(h_gated)
        return h_gated

    def prefix(self, h_gated):
        """
        h_gated: (B, d_state)
        returns: prefix: (B, n_prefix, dec_hidden)
        """
        B = h_gated.size(0)
        x = self.proj(h_gated)  # (B, n_prefix*dec_hidden)
        return x.view(B, self.n_prefix, -1)

    def forward(self, h, q_text_emb):
        """
        returns: (h_gated, prefix)
          h_gated: (B, d_state)
          prefix : (B, n_prefix, dec_hidden)
        """
        h_g = self.gate(h, q_text_emb)
        pfx = self.prefix(h_g)
        return h_g, pfx


In [17]:
# =========================================================
# CELL 15 — Full Multi-Task Model Wrapper (encoder shared; head losses task-isolated)
#   Change (1) integrated here:
#     - Apply FiLM gating on h_final using q_text_emb via BridgeAdapter
#     - Use gated h for user/item heads AND generation prefix adapter
#   All existing masks, losses, and execution flow preserved.
# =========================================================

class UniHisMultiTaskModel(nn.Module):
    def __init__(self, encoder, n_users, n_nodes, decoder, n_prefix=8, user_neg=64, d_qtext=768):
        super().__init__()
        self.encoder = encoder

        # Heads
        self.user_ranker = UserBilinearRanker(encoder.d_state, n_users)
        self.item_head = ItemMLPHead(encoder.d_state, n_nodes)
        self.bridge = BridgeAdapter(encoder.d_state, decoder.config.n_embd, d_qtext=d_qtext, n_prefix=n_prefix)

        # Decoder is external but registered for joint training
        self.decoder = decoder
        self.user_neg = user_neg

        # Losses
        self.item_ce = nn.CrossEntropyLoss()

    def forward(self, batch, loss_weights=None):
        """
        loss_weights: dict with keys in {"enc_action","enc_node","enc_task","user","item","gen"}.
        """
        if loss_weights is None:
            loss_weights = {
                "enc_action": 1.0,
                "enc_node": 1.0,
                "enc_task": 1.0,
                "user": 1.0,
                "item": 1.0,
                "gen": 1.0
            }

        enc = self.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h_raw = enc["h_final"]  # (B, d_state)

        # ---------- FiLM gating using query text embedding ----------
        h, prefix = self.bridge(h_raw, batch["q_text_emb"])  # gated h + prefix

        # ---------- Head losses (task-isolated by masks) ----------
        loss_user = torch.tensor(0.0, device=h.device)
        if float(batch["user_mask"].item()) > 0:
            loss_user = sampled_user_loss(self.user_ranker, h, batch["user_labels"], n_neg=self.user_neg)

        loss_item = torch.tensor(0.0, device=h.device)
        if float(batch["item_mask"].item()) > 0:
            logits_item = self.item_head(h)               # (B, n_nodes)
            loss_item = self.item_ce(logits_item, batch["item_labels"])

        loss_gen = torch.tensor(0.0, device=h.device)
        if float(batch["gen_mask"].item()) > 0:
            input_ids = batch["gen_input_ids"]            # (B=1, L)
            attn = batch["gen_attn"]                      # (B=1, L)

            # embeddings
            tok_emb = self.decoder.transformer.wte(input_ids)  # (B,L,dec_h)

            inputs_embeds = torch.cat([prefix, tok_emb], dim=1)  # (B,P+L,dec_h)

            # labels aligned with inputs_embeds length
            labels = torch.cat(
                [
                    torch.full((input_ids.size(0), prefix.size(1)), IGNORE_INDEX, device=h.device, dtype=torch.long),
                    input_ids
                ],
                dim=1
            )

            attn2 = torch.cat(
                [
                    torch.ones((attn.size(0), prefix.size(1)), device=h.device, dtype=attn.dtype),
                    attn
                ],
                dim=1
            )

            out = self.decoder(
                inputs_embeds=inputs_embeds,
                attention_mask=attn2,
                labels=labels
            )
            loss_gen = out.loss

        # ---------- Total ----------
        loss_enc = (
            loss_weights["enc_action"] * enc["loss_action"] +
            loss_weights["enc_node"]   * enc["loss_node"] +
            loss_weights["enc_task"]   * enc["loss_task"]
        )

        loss = (
            loss_enc +
            loss_weights["user"] * loss_user +
            loss_weights["item"] * loss_item +
            loss_weights["gen"]  * loss_gen
        )

        return {
            "loss": loss,
            "loss_enc": loss_enc,
            "loss_user": loss_user,
            "loss_item": loss_item,
            "loss_gen": loss_gen,
            "enc": enc,
            "h_raw": h_raw,
            "h": h
        }


In [18]:
# =========================================================
# CELL 16 — Instantiate + Optimizer
# =========================================================

encoder = UniHisSSMEncoder(
    vocab_sizes={"actions": len(actions2idx), "nodes": len(node2idx)},
    d_model=D_MODEL,
    d_state=64,
    n_tasks=n_tasks,
    d_qtext=D_MODEL,
    known_task=False
).to(device)

model = UniHisMultiTaskModel(
    encoder=encoder,
    n_users=len(user2idx_small),
    n_nodes=len(node2idx),
    decoder=dec,
    n_prefix=8,
    user_neg=64,
    d_qtext=D_MODEL
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# =========================================================
# PARAM COUNT — total + trainable + per-module breakdown
# =========================================================

def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_params(model)
print(f"TOTAL params     : {total:,}")
print(f"TRAINABLE params : {trainable:,}")

# ---- optional: breakdown by top-level submodules ----
print("\n--- Breakdown (top-level) ---")
for name, sub in model.named_children():
    t, tr = count_params(sub)
    print(f"{name:20s}  total={t:,}  trainable={tr:,}")

# ---- optional: more detailed breakdown (2-level) ----
print("\n--- Breakdown (2-level) ---")
for name, sub in model.named_children():
    for name2, sub2 in sub.named_children():
        t, tr = count_params(sub2)
        print(f"{name}.{name2:16s}  total={t:,}  trainable={tr:,}")


TOTAL params     : 130,158,225
TRAINABLE params : 130,158,225

--- Breakdown (top-level) ---
encoder               total=24,175,633  trainable=24,175,633
user_ranker           total=293,824  trainable=293,824
item_head             total=23,274,112  trainable=23,274,112
bridge                total=502,080  trainable=502,080
decoder               total=81,912,576  trainable=81,912,576
item_ce               total=0  trainable=0

--- Breakdown (2-level) ---
encoder.criterion         total=0  trainable=0
encoder.task_criterion    total=0  trainable=0
encoder.action_proj       total=3,840  trainable=3,840
encoder.MLP_mood          total=590,976  trainable=590,976
encoder.W_a               total=49,216  trainable=49,216
encoder.Mixer_L           total=4,160  trainable=4,160
encoder.Mixer_S           total=4,160  trainable=4,160
encoder.Mixer_E           total=4,160  trainable=4,160
encoder.W_fuse            total=579  trainable=579
encoder.Mixer_Fuse        total=4,160  trainable=4,160
encode

In [ ]:
# =========================================================
# CELL 17+ (SINGLE CELL) — TRAIN + OPTIONAL FINETUNE + LOAD-LATEST + EVALUATIONS
#
# What you get in ONE CELL:
#   A) Crash-safe training with intra-epoch row resume
#   B) Optional finetune phase chosen at run start:
#        - u : encoder + user head
#        - i : encoder + item head
#        - g : encoder + generation head
#      You will be prompted for finetune mode + finetune epochs (only if NO ckpt exists).
#   C) If finetune is not chosen (blank/0), continues MULTI-TASK training as-is.
#   D) Hard-load latest checkpoint before evaluation.
#   E) Runs evaluations (ITEM, GEN metrics + CSVs, USER, 100-way next-response)
#
# Requirements satisfied:
#   - Row checkpoints store row index; resume continues from NEXT row.
#   - End-of-epoch ckpt forces next-epoch resume.
#   - If no ckpt exists: bootstrap from FINETUNE_FROM only if it exists; else start fresh.
#   - No placeholders, no TODO, no manual patching, no omitted sections.
#
# Assumes earlier cells already defined:
#   model, optimizer, device, train_df, test_df, build_row_tensors, SEED,
#   USER_TASK, ITEM_TASK, GEN_TASKS (or task_names to derive it), node2idx, tok (tokenizer) etc.
#   If some are not defined earlier, this cell defines safe fallbacks below.
# =========================================================

import os
import re
import math
import time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# -----------------------------
# SAFE FALLBACK CONSTANTS (do not override if already defined)
# -----------------------------
if "USER_TASK" not in globals():
    USER_TASK = "next_user_prediction"
if "ITEM_TASK" not in globals():
    ITEM_TASK = "next_item_prediction"
if "DIALOG_TASK" not in globals():
    DIALOG_TASK = "dialogue_generation"
if "IGNORE_INDEX" not in globals():
    IGNORE_INDEX = -100

# GEN_TASKS fallback: use existing if present; else infer from task_names if present; else empty set
if "GEN_TASKS" not in globals():
    if "task_names" in globals():
        GEN_TASKS = set([t for t in task_names if ("summar" in str(t).lower()) or ("dialogue_generation" in str(t).lower())])
    else:
        GEN_TASKS = set()

# -----------------------------
# CONFIG
# -----------------------------
EPOCHS = 3
CKPT_DIR = "checkpoints_multitask"
os.makedirs(CKPT_DIR, exist_ok=True)

# If you want to ignore existing ckpts and force bootstrap/fresh plan, set this True.
FORCE_BOOTSTRAP = False

# Bootstrap checkpoint (used ONLY when no multitask ckpt is found or FORCE_BOOTSTRAP=True)
FINETUNE_FROM = "/teamspace/studios/this_studio/codes/checkpoints_multitask/epoch098_row000000.pt"
BOOTSTRAP_EPOCH = 98
BOOTSTRAP_ROW = 0

# Validation
VAL_PER_TASK = 500

# Row checkpoint frequency
SAVE_EVERY = 15000

# -----------------------------
# Checkpoint naming
# -----------------------------
ROW_RE   = re.compile(r"^epoch(\d+)_row(\d+)\.pt$")
EPOCH_RE = re.compile(r"^epoch(\d+)_EPOCH\.pt$")

def ckpt_row_path(epoch, row):
    return os.path.join(CKPT_DIR, f"epoch{epoch:03d}_row{row:06d}.pt")

def ckpt_epoch_path(epoch):
    return os.path.join(CKPT_DIR, f"epoch{epoch:03d}_EPOCH.pt")

def find_latest_ckpt():
    """
    Returns (kind, filename, epoch, row_or_none)
      kind ∈ {"epoch", "row"} where "epoch" outranks "row" within same epoch.
    """
    files = [f for f in os.listdir(CKPT_DIR) if f.endswith(".pt")]
    if not files:
        return None

    parsed = []
    for f in files:
        m = ROW_RE.match(f)
        if m:
            parsed.append(("row", f, int(m.group(1)), int(m.group(2))))
            continue
        m = EPOCH_RE.match(f)
        if m:
            parsed.append(("epoch", f, int(m.group(1)), None))
            continue

    if not parsed:
        return None

    def key(x):
        kind, _, e, r = x
        kind_rank = 1 if kind == "epoch" else 0  # epoch > row within same epoch
        row_rank = -1 if r is None else r
        return (e, kind_rank, row_rank)

    parsed.sort(key=key)
    return parsed[-1]

# -----------------------------
# Loss weights (MULTI-TASK default; preserved)
# -----------------------------
LOSS_W_MT = {
    "enc_action": 1.0,
    "enc_node": 1.0,
    "enc_task": 2.0,
    "user": 4.0,
    "item": 4.0,
    "gen": 4.0
}

# -----------------------------
# Finetune weights (mode-specific; encoder always on)
# -----------------------------
LOSS_W_FT_U = {
    "enc_action": 1.0,
    "enc_node": 1.0,
    "enc_task": 2.0,
    "user": 8.0,
    "item": 0.0,
    "gen": 0.0
}

LOSS_W_FT_I = {
    "enc_action": 1.0,
    "enc_node": 1.0,
    "enc_task": 2.0,
    "user": 0.0,
    "item": 8.0,
    "gen": 0.0
}

LOSS_W_FT_G = {
    "enc_action": 1.0,
    "enc_node": 1.0,
    "enc_task": 2.0,
    "user": 0.0,
    "item": 0.0,
    "gen": 12.0
}

# -----------------------------
# Deterministic epoch shuffle
# -----------------------------
def make_epoch_df(df, epoch, base_seed=SEED):
    rs = int(base_seed) + int(epoch)
    return df.sample(frac=1, random_state=rs).reset_index(drop=True)

# -----------------------------
# Validation: fixed 500 per task (deterministic)
# -----------------------------
TASKS = sorted(train_df["task"].astype(str).str.strip().unique().tolist())

def build_val_sets(df, base_seed=SEED):
    out = {}
    for t in TASKS:
        sub = df[df["task"].astype(str).str.strip() == t]
        n = min(VAL_PER_TASK, len(sub))
        out[t] = sub.sample(n=n, random_state=base_seed).reset_index(drop=True)
    return out

VAL_SETS = build_val_sets(train_df, base_seed=SEED)

@torch.no_grad()
def run_validation(model, val_sets, device, loss_weights):
    model.eval()
    per_task = {}
    overall_tot = defaultdict(float)
    overall_n = 0

    for task, dfv in val_sets.items():
        tot = defaultdict(float)
        ok = 0
        for i in range(len(dfv)):
            b = build_row_tensors(dfv.iloc[i], device)
            if not b["ok"]:
                continue
            out = model(b, loss_weights=loss_weights)
            L = out["loss"]
            if not torch.isfinite(L):
                continue

            ok += 1
            tot["loss"]      += float(out["loss"].item())
            tot["loss_enc"]  += float(out["loss_enc"].item())
            tot["loss_user"] += float(out["loss_user"].item())
            tot["loss_item"] += float(out["loss_item"].item())
            tot["loss_gen"]  += float(out["loss_gen"].item())

        per_task[task] = {
            "count": ok,
            "loss":      tot["loss"]      / max(1, ok),
            "loss_enc":  tot["loss_enc"]  / max(1, ok),
            "loss_user": tot["loss_user"] / max(1, ok),
            "loss_item": tot["loss_item"] / max(1, ok),
            "loss_gen":  tot["loss_gen"]  / max(1, ok),
        }

        for k in ["loss", "loss_enc", "loss_user", "loss_item", "loss_gen"]:
            overall_tot[k] += tot[k]
        overall_n += ok

    overall = {
        "count": overall_n,
        "loss":      overall_tot["loss"]      / max(1, overall_n),
        "loss_enc":  overall_tot["loss_enc"]  / max(1, overall_n),
        "loss_user": overall_tot["loss_user"] / max(1, overall_n),
        "loss_item": overall_tot["loss_item"] / max(1, overall_n),
        "loss_gen":  overall_tot["loss_gen"]  / max(1, overall_n),
    }

    model.train()
    return per_task, overall

# =========================================================
# RESUME / BOOTSTRAP / FRESH START + FINETUNE PLAN
# =========================================================
start_epoch, start_row = 0, 0
plan = {
    "enabled": False,
    "mode": "",
    "epochs": 0,
    "base_epoch": 0,   # first epoch index that finetune applies to
}

latest = None if FORCE_BOOTSTRAP else find_latest_ckpt()

if latest is not None:
    kind, fname, e, r = latest
    ck = torch.load(os.path.join(CKPT_DIR, fname), map_location=device, weights_only=False)

    model.load_state_dict(ck["model"], strict=True)
    optimizer.load_state_dict(ck["optim"])

    # Restore plan if present; else default to "no finetune"
    if isinstance(ck, dict) and ("plan" in ck) and isinstance(ck["plan"], dict):
        plan = ck["plan"]

    if kind == "epoch":
        start_epoch = int(e) + 1
        start_row = 0
    else:
        start_epoch = int(e)
        start_row = int(r) + 1

    print(f"RESUME: {kind.upper()} ckpt {fname} -> epoch={start_epoch}, row={start_row}")
    if plan.get("enabled", False):
        print(f"RESUME PLAN: finetune enabled mode={plan.get('mode')} epochs={plan.get('epochs')} base_epoch={plan.get('base_epoch')}")
    else:
        print("RESUME PLAN: no finetune (multi-task only)")

else:
    # No ckpt exists: ask for finetune mode + epochs as requested
    # (robust to non-interactive environments via env vars)
    ft_mode_env = os.environ.get("FINETUNE_MODE", "").strip().lower()
    ft_epochs_env = os.environ.get("FINETUNE_EPOCHS", "").strip()

    ft_mode = ft_mode_env
    if ft_mode == "":
        try:
            ft_mode = input("Finetune mode? [u=enc+user, i=enc+item, g=enc+gen, blank=none] : ").strip().lower()
        except Exception:
            ft_mode = ""

    if ft_mode not in {"", "u", "i", "g"}:
        print(f"Invalid finetune mode '{ft_mode}'. Finetune disabled.")
        ft_mode = ""

    ft_epochs = 0
    if ft_mode != "":
        if ft_epochs_env != "":
            try:
                ft_epochs = int(ft_epochs_env)
            except Exception:
                ft_epochs = 0
        else:
            try:
                ft_epochs = int(input(f"Finetune epochs for mode '{ft_mode}' (integer >=0): ").strip())
            except Exception:
                ft_epochs = 0

        if ft_epochs < 0:
            ft_epochs = 0

    # Bootstrap if FINETUNE_FROM exists; else start fresh (as you requested)
    if os.path.exists(FINETUNE_FROM):
        ck = torch.load(FINETUNE_FROM, map_location=device, weights_only=False)
        src_sd = ck["model"] if isinstance(ck, dict) and ("model" in ck) else ck
        tgt_sd = model.state_dict()

        kept = {}
        skipped_missing = 0
        skipped_shape = 0
        for k, v in src_sd.items():
            if k not in tgt_sd:
                skipped_missing += 1
                continue
            if tuple(v.shape) != tuple(tgt_sd[k].shape):
                skipped_shape += 1
                continue
            kept[k] = v

        missing, unexpected = model.load_state_dict(kept, strict=False)

        print("BOOTSTRAP LOAD REPORT")
        print("  source keys:", len(src_sd))
        print("  loaded keys:", len(kept))
        print("  skipped missing keys:", skipped_missing)
        print("  skipped shape-mismatch keys:", skipped_shape)
        print("  missing after load:", len(missing))
        print("  unexpected after load:", len(unexpected))

        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
        start_epoch = int(BOOTSTRAP_EPOCH)
        start_row = int(BOOTSTRAP_ROW)
        print(f"BOOTSTRAP: epoch={start_epoch}, row={start_row}")

    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
        start_epoch = 0
        start_row = 0
        print("NO CHECKPOINT FOUND — STARTING FRESH")

    # Finalize plan base_epoch after we know where we start
    plan = {
        "enabled": (ft_mode != "" and ft_epochs > 0),
        "mode": ft_mode,
        "epochs": int(ft_epochs),
        "base_epoch": int(start_epoch),
    }

    if plan["enabled"]:
        print(f"PLAN: finetune enabled mode={plan['mode']} epochs={plan['epochs']} starting at epoch={plan['base_epoch']}")
    else:
        print("PLAN: no finetune (multi-task only)")

# -----------------------------
# Build finetune dataset based on plan mode
# -----------------------------
def get_phase_and_weights_and_df(epoch, plan):
    """
    Returns: (phase_str, loss_weights_dict, epoch_df)
      phase_str in {"ft_u","ft_i","ft_g","mt"}
    """
    if not plan.get("enabled", False):
        return "mt", LOSS_W_MT, train_df

    base = int(plan.get("base_epoch", 0))
    n_ft = int(plan.get("epochs", 0))
    mode = str(plan.get("mode", "")).strip().lower()

    if n_ft <= 0 or mode == "":
        return "mt", LOSS_W_MT, train_df

    if base <= epoch <= (base + n_ft - 1):
        if mode == "u":
            df_ft = train_df[train_df["task"].astype(str).str.strip() == USER_TASK].reset_index(drop=True)
            return "ft_u", LOSS_W_FT_U, df_ft
        if mode == "i":
            df_ft = train_df[train_df["task"].astype(str).str.strip() == ITEM_TASK].reset_index(drop=True)
            return "ft_i", LOSS_W_FT_I, df_ft
        if mode == "g":
            df_ft = train_df[train_df["task"].astype(str).str.strip().isin(list(GEN_TASKS))].reset_index(drop=True)
            return "ft_g", LOSS_W_FT_G, df_ft

    return "mt", LOSS_W_MT, train_df

# =========================================================
# TRAIN (with optional finetune phase embedded)
# =========================================================
for epoch in range(start_epoch, EPOCHS):
    model.train()
    tot = defaultdict(float)
    ok = 0

    phase, LOSS_W_ACTIVE, base_df = get_phase_and_weights_and_df(epoch, plan)

    if len(base_df) == 0:
        print(f"[WARN] Phase {phase} has 0 rows. Falling back to multi-task for epoch {epoch}.")
        phase, LOSS_W_ACTIVE, base_df = "mt", LOSS_W_MT, train_df

    epoch_df = make_epoch_df(base_df, epoch, base_seed=SEED)

    # exact intra-epoch resume only for the first epoch after loading ckpt
    row_start = start_row if epoch == start_epoch else 0
    if row_start < 0:
        row_start = 0
    if row_start > len(epoch_df):
        row_start = len(epoch_df)

    pbar = tqdm(
        range(row_start, len(epoch_df)),
        desc=f"[{phase}] Epoch {epoch+1}/{EPOCHS}",
        total=len(epoch_df),
        initial=row_start
    )

    for i in pbar:
        batch = build_row_tensors(epoch_df.iloc[i], device)
        if not batch["ok"]:
            continue

        out = model(batch, loss_weights=LOSS_W_ACTIVE)
        loss = out["loss"]
        if not torch.isfinite(loss):
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        ok += 1
        tot["loss"]      += float(out["loss"].item())
        tot["loss_enc"]  += float(out["loss_enc"].item())
        tot["loss_user"] += float(out["loss_user"].item())
        tot["loss_item"] += float(out["loss_item"].item())
        tot["loss_gen"]  += float(out["loss_gen"].item())

        # intra-epoch row checkpoint
        if (i % SAVE_EVERY == 0) or (i == len(epoch_df) - 1):
            torch.save(
                {
                    "kind": "row",
                    "phase": phase,
                    "epoch": epoch,
                    "row": i,
                    "plan": plan,
                    "model": model.state_dict(),
                    "optim": optimizer.state_dict(),
                },
                ckpt_row_path(epoch, i)
            )

        if ok > 0:
            pbar.set_postfix(
                avg_loss=tot["loss"]/ok,
                enc=tot["loss_enc"]/ok,
                user=tot["loss_user"]/ok,
                item=tot["loss_item"]/ok,
                gen=tot["loss_gen"]/ok
            )

    # end-of-epoch checkpoint
    torch.save(
        {
            "kind": "epoch",
            "phase": phase,
            "epoch": epoch,
            "plan": plan,
            "model": model.state_dict(),
            "optim": optimizer.state_dict(),
        },
        ckpt_epoch_path(epoch)
    )

    # validation (use active weights to reflect current phase behavior)
    val_per_task, val_overall = run_validation(model, VAL_SETS, device, LOSS_W_ACTIVE)

    print(
        f"\n[{phase}] Epoch {epoch+1}/{EPOCHS} TRAIN: "
        f"avg_loss={tot['loss']/max(1,ok):.6f} "
        f"enc={tot['loss_enc']/max(1,ok):.6f} "
        f"user={tot['loss_user']/max(1,ok):.6f} "
        f"item={tot['loss_item']/max(1,ok):.6f} "
        f"gen={tot['loss_gen']/max(1,ok):.6f} "
        f"(ok={ok})"
    )

    print(
        f"[{phase}] Epoch {epoch+1}/{EPOCHS} VAL OVERALL: "
        f"loss={val_overall['loss']:.6f} "
        f"enc={val_overall['loss_enc']:.6f} "
        f"user={val_overall['loss_user']:.6f} "
        f"item={val_overall['loss_item']:.6f} "
        f"gen={val_overall['loss_gen']:.6f} "
        f"(n={val_overall['count']})"
    )

    for t in TASKS:
        v = val_per_task.get(t, None)
        if v is None:
            continue
        print(
            f"  VAL[{t}] (n={v['count']}): "
            f"loss={v['loss']:.6f} "
            f"enc={v['loss_enc']:.6f} "
            f"user={v['loss_user']:.6f} "
            f"item={v['loss_item']:.6f} "
            f"gen={v['loss_gen']:.6f}"
        )

    # after the first resumed epoch completes, all future epochs start at row 0
    start_row = 0



RESUME: ROW ckpt epoch000_row000000.pt -> epoch=0, row=1
RESUME PLAN: no finetune (multi-task only)


[mt] Epoch 1/3:   0%|          | 1/78274 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



[mt] Epoch 1/3 TRAIN: avg_loss=22.518796 enc=14.692093 user=0.241477 item=0.347838 gen=1.367361 (ok=77925)
[mt] Epoch 1/3 VAL OVERALL: loss=30.307188 enc=19.599360 user=1.043114 item=0.035929 gen=1.597915 (n=1995)
  VAL[dialogue_generation] (n=500): loss=45.565140 enc=26.263424 user=0.000000 item=0.000000 gen=4.825429
  VAL[next_item_prediction] (n=498): loss=13.084743 enc=12.509012 user=0.000000 item=0.143933 gen=0.000000
  VAL[next_user_prediction] (n=500): loss=43.084502 enc=26.436409 user=4.162023 item=0.000000 gen=0.000000
  VAL[personalized_summarization] (n=497): loss=19.359794 enc=13.121364 user=0.000000 item=0.000000 gen=1.559607


[mt] Epoch 2/3:   0%|          | 0/78274 [00:00<?, ?it/s]


[mt] Epoch 2/3 TRAIN: avg_loss=20.136346 enc=14.946822 user=0.226515 item=0.064621 gen=1.006245 (ok=77926)
[mt] Epoch 2/3 VAL OVERALL: loss=29.095545 enc=19.611519 user=0.893578 item=0.014724 gen=1.462705 (n=1995)
  VAL[dialogue_generation] (n=500): loss=44.739904 enc=26.472752 user=0.000000 item=0.000000 gen=4.566788
  VAL[next_item_prediction] (n=498): loss=13.042710 enc=12.806770 user=0.000000 item=0.058985 gen=0.000000
  VAL[next_user_prediction] (n=500): loss=40.497974 enc=26.236472 user=3.565375 item=0.000000 gen=0.000000
  VAL[personalized_summarization] (n=497): loss=17.970632 enc=12.862369 user=0.000000 item=0.000000 gen=1.277066


[mt] Epoch 3/3:   0%|          | 0/78274 [00:00<?, ?it/s]

In [ ]:
# =========================================================
# LOAD LATEST CKPT BEFORE EVALUATION (hard ensure)
# =========================================================
latest_eval = find_latest_ckpt()
if latest_eval is None:
    raise FileNotFoundError(f"No checkpoints found in {CKPT_DIR}")

kind, fname, e, r = latest_eval
path = os.path.join(CKPT_DIR, fname)
ck = torch.load(path, map_location=device, weights_only=False)

model.load_state_dict(ck["model"], strict=True)
model.eval()

print(f"\nLoaded for eval: {fname} (kind={kind}, epoch={e}, row={r}, phase={ck.get('phase','?')})")

# =========================================================
# EVALUATION 1 — ITEM (next_item_prediction), 0–100 scale
# =========================================================
K = 15
item_eval_df = test_df[test_df["task"].astype(str).str.strip() == ITEM_TASK].reset_index(drop=True)
item_eval_df = item_eval_df[:50]
print("\nItem eval rows:", len(item_eval_df))

mrrs = []
ndcg5 = []
ndcg10 = []
n_eval = 0

with torch.no_grad():
    pbar = tqdm(range(len(item_eval_df)), desc="Evaluating ITEM")
    for i in pbar:
        batch = build_row_tensors(item_eval_df.iloc[i], device)
        if not batch["ok"]:
            continue

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )
        h = enc["h_final"]  # (1,d)

        logits = model.item_head(h)[0]  # (n_nodes,)
        topk = torch.topk(logits, K).indices.tolist()

        gold = int(batch["item_labels"].item())

        rank = None
        for rr, idx in enumerate(topk, 1):
            if idx == gold:
                rank = rr
                break

        mrrs.append(0.0 if rank is None else 1.0 / rank)
        ndcg5.append((1.0 / math.log2(rank + 1)) if rank is not None and rank <= 5 else 0.0)
        ndcg10.append((1.0 / math.log2(rank + 1)) if rank is not None and rank <= 10 else 0.0)

        n_eval += 1
        if n_eval > 0:
            pbar.set_postfix(
                MRR=100.0 * np.mean(mrrs),
                N5=100.0 * np.mean(ndcg5),
                N10=100.0 * np.mean(ndcg10)
            )

print("======================================")
print("ITEM EVAL COUNT:", n_eval)
print(f"MRR@{K}  : {100.0 * float(np.mean(mrrs)):.4f}")
print(f"NDCG@5   : {100.0 * float(np.mean(ndcg5)):.4f}")
print(f"NDCG@10  : {100.0 * float(np.mean(ndcg10)):.4f}")
print("======================================")

# =========================================================
# EVALUATION 2 — GENERATION (metrics + CSVs)
#   Metrics: ROUGE-SU4, ROUGE-L, ROUGE-2, METEOR, BLEU-4
# =========================================================

# -----------------------------
# Tokenization (deterministic)
# -----------------------------
def _tok(s: str):
    s = "" if s is None else str(s)
    s = s.strip().lower()
    if s == "":
        return []
    return s.split()

def _ngrams(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

# -----------------------------
# ROUGE-2 (F1 on bigrams)
# -----------------------------
def rouge2_f1(pred: str, ref: str):
    p = _tok(pred)
    r = _tok(ref)
    p2 = _ngrams(p, 2)
    r2 = _ngrams(r, 2)
    if len(p2) == 0 and len(r2) == 0:
        return 1.0
    if len(p2) == 0 or len(r2) == 0:
        return 0.0
    cp = Counter(p2)
    cr = Counter(r2)
    overlap = sum((cp & cr).values())
    prec = overlap / max(1, len(p2))
    rec  = overlap / max(1, len(r2))
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

# -----------------------------
# ROUGE-L (LCS-based F1)
# -----------------------------
def _lcs_len(a, b):
    n, m = len(a), len(b)
    if n == 0 or m == 0:
        return 0
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        ai = a[i - 1]
        for j in range(1, m + 1):
            tmp = dp[j]
            if ai == b[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return dp[m]

def rougeL_f1(pred: str, ref: str):
    p = _tok(pred)
    r = _tok(ref)
    if len(p) == 0 and len(r) == 0:
        return 1.0
    if len(p) == 0 or len(r) == 0:
        return 0.0
    lcs = _lcs_len(p, r)
    prec = lcs / max(1, len(p))
    rec  = lcs / max(1, len(r))
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

# -----------------------------
# ROUGE-SU4 (unigrams + skip-bigrams with max skip=4), F1
# -----------------------------
def _skip_bigrams(tokens, max_skip=4):
    sb = []
    L = len(tokens)
    for i in range(L):
        for j in range(i+1, min(L, i+max_skip+2)):
            sb.append((tokens[i], tokens[j]))
    return sb

def rougeSU4_f1(pred: str, ref: str, max_skip=4):
    p = _tok(pred)
    r = _tok(ref)

    pu = Counter(p)
    ru = Counter(r)

    psb = Counter(_skip_bigrams(p, max_skip=max_skip))
    rsb = Counter(_skip_bigrams(r, max_skip=max_skip))

    ou = sum((pu & ru).values())
    osb = sum((psb & rsb).values())

    tot_p = len(p) + sum(psb.values())
    tot_r = len(r) + sum(rsb.values())

    if tot_p == 0 and tot_r == 0:
        return 1.0
    if tot_p == 0 or tot_r == 0:
        return 0.0

    overlap = ou + osb
    prec = overlap / max(1, tot_p)
    rec  = overlap / max(1, tot_r)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

# -----------------------------
# BLEU-4 (sentence BLEU with smoothing)
# -----------------------------
def bleu4_sentence(pred: str, ref: str):
    p = _tok(pred)
    r = _tok(ref)
    if len(p) == 0 and len(r) == 0:
        return 1.0
    if len(p) == 0 or len(r) == 0:
        return 0.0

    bp = 1.0
    if len(p) < len(r):
        bp = math.exp(1.0 - (len(r) / max(1, len(p))))

    precisions = []
    for n in [1, 2, 3, 4]:
        p_ngr = Counter(_ngrams(p, n))
        r_ngr = Counter(_ngrams(r, n))
        total = sum(p_ngr.values())
        overlap = sum((p_ngr & r_ngr).values())
        pn_smooth = (overlap + 1.0) / (total + 1.0) if total > 0 else 0.0
        precisions.append(pn_smooth)

    geo_mean = math.exp(sum(math.log(max(1e-12, x)) for x in precisions) / 4.0)
    return bp * geo_mean

# -----------------------------
# METEOR (NLTK official if available; else strict fallback)
# -----------------------------
try:
    from nltk.translate.meteor_score import meteor_score as _nltk_meteor_score
    _HAS_NLTK_METEOR = True
except Exception:
    _HAS_NLTK_METEOR = False

def meteor_sentence(pred: str, ref: str):
    p = _tok(pred)
    r = _tok(ref)
    if len(p) == 0 and len(r) == 0:
        return 1.0
    if len(p) == 0 or len(r) == 0:
        return 0.0

    if _HAS_NLTK_METEOR:
        try:
            return float(_nltk_meteor_score([r], p))
        except Exception:
            pass

    cp = Counter(p)
    cr = Counter(r)
    overlap = sum((cp & cr).values())
    prec = overlap / max(1, len(p))
    rec  = overlap / max(1, len(r))
    if prec + rec == 0:
        return 0.0
    fmean = (10 * prec * rec) / (rec + 9 * prec)

    r_pos = defaultdict(list)
    for i, tok_ in enumerate(r):
        r_pos[tok_].append(i)
    used = set()
    aligned = []
    for i, tok_ in enumerate(p):
        if tok_ in r_pos:
            for j in r_pos[tok_]:
                if j not in used:
                    used.add(j)
                    aligned.append((i, j))
                    break
    aligned.sort()
    chunks = 0
    for k in range(len(aligned)):
        if k == 0:
            chunks = 1
        else:
            prev = aligned[k-1]
            cur = aligned[k]
            if not (cur[0] == prev[0] + 1 and cur[1] == prev[1] + 1):
                chunks += 1
    penalty = 0.5 * ((chunks / max(1, overlap)) ** 3) if overlap > 0 else 1.0
    return (1 - penalty) * fmean

# -----------------------------
# Prefix-conditioned decoding (BOS warm-start + sampling)
# -----------------------------
@torch.no_grad()
def generate_with_prefix_sampling(decoder, tokenizer, prefix_embeds, max_new_tokens=64, temperature=0.1, top_k=20):
    decoder.eval()
    B = prefix_embeds.size(0)
    device_ = prefix_embeds.device

    eos_id = tokenizer.eos_token_id
    bos_id = tokenizer.bos_token_id
    if bos_id is None:
        bos_id = eos_id

    if bos_id is None:
        raise ValueError("Tokenizer has neither bos_token_id nor eos_token_id. Set one.")

    bos = torch.full((B, 1), int(bos_id), device=device_, dtype=torch.long)
    bos_emb = decoder.transformer.wte(bos)

    init_embeds = torch.cat([prefix_embeds, bos_emb], dim=1)
    out = decoder(inputs_embeds=init_embeds, use_cache=True)
    past = out.past_key_values

    logits = out.logits[:, -1, :]

    generated = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device_)

    for _ in range(max_new_tokens):
        step_logits = logits / max(1e-8, float(temperature))

        if top_k is not None and int(top_k) > 0:
            k = min(int(top_k), step_logits.size(-1))
            vals, idxs = torch.topk(step_logits, k, dim=-1)
            filt = torch.full_like(step_logits, float("-inf"))
            filt.scatter_(1, idxs, vals)
            step_logits = filt

        probs = torch.softmax(step_logits, dim=-1)
        next_ids = torch.multinomial(probs, num_samples=1).squeeze(-1)

        for b in range(B):
            if not bool(finished[b]):
                generated[b].append(int(next_ids[b].item()))

        if eos_id is not None:
            finished = finished | (next_ids == eos_id)
            if bool(finished.all()):
                break

        out = decoder(input_ids=next_ids.unsqueeze(1), past_key_values=past, use_cache=True)
        past = out.past_key_values
        logits = out.logits[:, -1, :]

    texts = [tokenizer.decode(ids, skip_special_tokens=True).strip() for ids in generated]
    return texts

# -----------------------------
# Build gen evaluation set
# -----------------------------
gen_eval_df = train_df[train_df["task"].astype(str).str.strip().isin(list(GEN_TASKS))].reset_index(drop=True)
gen_eval_df = gen_eval_df[:50]
print("\nGen eval rows:", len(gen_eval_df))

OUT_DIR = "gen_eval_results"
os.makedirs(OUT_DIR, exist_ok=True)
stamp = time.strftime("%Y%m%d_%H%M%S")

rows = []
model.eval()

with torch.no_grad():
    pbar = tqdm(range(len(gen_eval_df)), desc="Autoregressive GEN eval")
    for i in pbar:
        batch = build_row_tensors(gen_eval_df.iloc[i], device)
        if not batch["ok"]:
            continue

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )
        h = enc["h_final"]

        ref = str(batch["gen_text"]).strip()
        prefix = model.bridge(h)

        pred = generate_with_prefix_sampling(
            model.decoder, tok, prefix,
            max_new_tokens=64, temperature=0.9, top_k=50
        )[0]

        rows.append({
            "task": batch["task"],
            "userid": batch["userid_raw"],
            "tid": batch["tid"],
            "ref": ref,
            "pred": pred,
            "rouge_su4_f1": rougeSU4_f1(pred, ref, max_skip=4),
            "rouge_l_f1": rougeL_f1(pred, ref),
            "rouge_2_f1": rouge2_f1(pred, ref),
            "meteor": meteor_sentence(pred, ref),
            "bleu4": bleu4_sentence(pred, ref)
        })

        if len(rows) > 0 and (len(rows) % 50 == 0):
            df_tmp = pd.DataFrame(rows)
            pbar.set_postfix(
                SU4=float(df_tmp["rouge_su4_f1"].mean()),
                RL=float(df_tmp["rouge_l_f1"].mean()),
                R2=float(df_tmp["rouge_2_f1"].mean()),
                MET=float(df_tmp["meteor"].mean()),
                BLEU=float(df_tmp["bleu4"].mean())
            )

df_all = pd.DataFrame(rows)
print("Processed gen examples:", len(df_all))

all_path = os.path.join(OUT_DIR, f"gen_per_example_{stamp}.csv")
df_all.to_csv(all_path, index=False)
print("Saved:", all_path)

agg_rows = []
for t, g in df_all.groupby("task"):
    t_path = os.path.join(OUT_DIR, f"gen_per_example_{t}_{stamp}.csv")
    g.to_csv(t_path, index=False)

    agg = {
        "task": t,
        "count": int(len(g)),
        "rouge_su4_f1": float(g["rouge_su4_f1"].mean()),
        "rouge_l_f1": float(g["rouge_l_f1"].mean()),
        "rouge_2_f1": float(g["rouge_2_f1"].mean()),
        "meteor": float(g["meteor"].mean()),
        "bleu4": float(g["bleu4"].mean()),
    }
    agg_rows.append(agg)

    print("\n---", t, "---")
    print("Saved:", t_path)
    print(pd.Series(agg))

df_agg = pd.DataFrame(agg_rows).sort_values("task").reset_index(drop=True)
agg_path = os.path.join(OUT_DIR, f"gen_aggregate_{stamp}.csv")
df_agg.to_csv(agg_path, index=False)
print("\nSaved:", agg_path)
print("\nAGGREGATE SUMMARY:\n", df_agg)
print("\nMETEOR backend:", "NLTK official" if _HAS_NLTK_METEOR else "STRICT FALLBACK (exact-match only)")

if len(df_all) > 0:
    print("\nGEN SAMPLE (ref/pred):")
    print(df_all[["ref", "pred"]].head(5))

# =========================================================
# EVALUATION 3 — USER (next_user_prediction), 0–100 scale
# =========================================================
K10 = 10
model.eval()

mrrs = []
recall1 = []
hit5 = []
hit10 = []
ndcg5 = []
ndcg10 = []
n_eval = 0

test_user_df = train_df[train_df["task"].astype(str).str.strip() == USER_TASK].reset_index(drop=True)
test_user_df = test_user_df[:5000]
print("\nUser eval rows:", len(test_user_df))

with torch.no_grad():
    pbar = tqdm(range(len(test_user_df)), desc="Evaluating USER")
    for i in pbar:
        batch = build_row_tensors(test_user_df.iloc[i], device)
        if not batch["ok"]:
            continue

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h = enc["h_final"]
        scores = model.user_ranker.scores_all(h)[0]
        top10 = torch.topk(scores, K10).indices.tolist()

        gold = int(batch["user_labels"].item())

        rank = None
        for rr, u in enumerate(top10, 1):
            if u == gold:
                rank = rr
                break

        mrrs.append(0.0 if rank is None else 1.0 / rank)
        recall1.append(1.0 if rank == 1 else 0.0)
        hit5.append(1.0 if rank is not None and rank <= 5 else 0.0)
        hit10.append(1.0 if rank is not None and rank <= 10 else 0.0)
        ndcg5.append((1.0 / math.log2(rank + 1)) if rank is not None and rank <= 5 else 0.0)
        ndcg10.append((1.0 / math.log2(rank + 1)) if rank is not None and rank <= 10 else 0.0)

        n_eval += 1
        if n_eval > 0:
            pbar.set_postfix(
                MRR=100.0 * np.mean(mrrs),
                R1=100.0 * np.mean(recall1),
                H5=100.0 * np.mean(hit5),
                H10=100.0 * np.mean(hit10),
                N5=100.0 * np.mean(ndcg5),
                N10=100.0 * np.mean(ndcg10)
            )

print("======================================")
print("USER EVAL COUNT:", n_eval)
print(f"MRR@10    : {100.0 * float(np.mean(mrrs)):.4f}")
print(f"Recall@1  : {100.0 * float(np.mean(recall1)):.4f}")
print(f"Hit@5     : {100.0 * float(np.mean(hit5)):.4f}")
print(f"Hit@10    : {100.0 * float(np.mean(hit10)):.4f}")
print(f"NDCG@5    : {100.0 * float(np.mean(ndcg5)):.4f}")
print(f"NDCG@10   : {100.0 * float(np.mean(ndcg10)):.4f}")
print("======================================")

# =========================================================
# EVALUATION 4 — Cross-task Next-Response Prediction (dialogue rows), 100-way
# =========================================================
CAND_K = 99
NEG_K = CAND_K - 1
EVAL_N = 500
K10, K5 = 10, 5

dlg_eval_df = test_df[test_df["task"].astype(str).str.strip() == DIALOG_TASK].reset_index(drop=True)
dlg_eval_df = dlg_eval_df[:5000]
if EVAL_N is not None and len(dlg_eval_df) > EVAL_N:
    dlg_eval_df = dlg_eval_df.sample(EVAL_N, random_state=SEED).reset_index(drop=True)

print("\nDialogue eval rows:", len(dlg_eval_df))

resp_pool = []
for nid in node2idx.keys():
    if str(nid).lower().startswith("r-"):
        resp_pool.append(nid)

if len(resp_pool) < CAND_K:
    raise ValueError(f"Response pool too small: {len(resp_pool)} < {CAND_K}. Check response ids in node2idx with prefix 'r-'.")

resp_pool = np.array(resp_pool, dtype=object)
print("Response pool size (from node2idx):", len(resp_pool))

def _safe_canonical_in_vocab(text_id):
    if text_id is None:
        return None
    s = str(text_id).strip()
    if s == "":
        return None
    if s in node2idx:
        return s
    return None

def sample_candidates(gold_id, pool, k_neg=NEG_K):
    if len(pool) <= k_neg + 1:
        negs = []
        while len(negs) < k_neg:
            c = pool[np.random.randint(0, len(pool))]
            if c != gold_id:
                negs.append(c)
        return [gold_id] + negs

    idx = np.random.choice(len(pool), size=k_neg + 5, replace=False)
    cand = pool[idx].tolist()
    cand = [c for c in cand if c != gold_id]
    cand = cand[:k_neg]
    while len(cand) < k_neg:
        c = pool[np.random.randint(0, len(pool))]
        if c != gold_id:
            cand.append(c)
    return [gold_id] + cand

def mrr_from_rank(rank):
    return 0.0 if rank is None else 1.0 / rank

def ndcg_from_rank(rank):
    return 0.0 if rank is None else 1.0 / math.log2(rank + 1)

model.eval()

mrrs = []
recall1 = []
hit5 = []
hit10 = []
ndcg5 = []
ndcg10 = []
n_eval = 0
n_skipped = 0

with torch.no_grad():
    pbar = tqdm(range(len(dlg_eval_df)), desc="Evaluating Next-Response (100-way)")
    for i in pbar:
        row = dlg_eval_df.iloc[i]
        batch = build_row_tensors(row, device)
        if not batch["ok"]:
            n_skipped += 1
            continue

        gold_tid = _safe_canonical_in_vocab(batch["tid"])
        if gold_tid is None:
            n_skipped += 1
            continue

        cands = sample_candidates(gold_tid, resp_pool, k_neg=NEG_K)
        cand_idx = [node2idx[c] for c in cands]

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )
        h = enc["h_final"]

        logits_all = model.item_head(h)[0]
        cand_scores = logits_all[cand_idx]

        order = torch.argsort(cand_scores, descending=True)
        ranked_pos = order.tolist()

        rank = None
        for rpos, j in enumerate(ranked_pos, 1):
            if j == 0:
                rank = rpos
                break

        mrrs.append(mrr_from_rank(rank))
        recall1.append(1.0 if rank == 1 else 0.0)
        hit5.append(1.0 if rank is not None and rank <= 5 else 0.0)
        hit10.append(1.0 if rank is not None and rank <= 10 else 0.0)
        ndcg5.append(ndcg_from_rank(rank) if rank is not None and rank <= 5 else 0.0)
        ndcg10.append(ndcg_from_rank(rank) if rank is not None and rank <= 10 else 0.0)

        n_eval += 1
        if n_eval > 0:
            pbar.set_postfix(
                MRR=100.0 * float(np.mean(mrrs)),
                R1=100.0 * float(np.mean(recall1)),
                H5=100.0 * float(np.mean(hit5)),
                H10=100.0 * float(np.mean(hit10)),
                N5=100.0 * float(np.mean(ndcg5)),
                N10=100.0 * float(np.mean(ndcg10)),
                skipped=n_skipped
            )

print("======================================")
print("NEXT-RESPONSE (100-way) EVAL")
print("Eval count :", n_eval)
print("Skipped    :", n_skipped)
print(f"MRR@{CAND_K}   : {100.0 * float(np.mean(mrrs)):.4f}")
print(f"Recall@1  : {100.0 * float(np.mean(recall1)):.4f}")
print(f"Hit@5     : {100.0 * float(np.mean(hit5)):.4f}")
print(f"Hit@10    : {100.0 * float(np.mean(hit10)):.4f}")
print(f"NDCG@5    : {100.0 * float(np.mean(ndcg5)):.4f}")
print(f"NDCG@10   : {100.0 * float(np.mean(ndcg10)):.4f}")
print("======================================")
